# 05 Model Training
Train multiple classification models on the feature set.

In [ ]:
from pathlib import Path
import json
import pandas as pd
from src.pipeline.engine import run_pipeline
from src.training.train_model import train_models
from src.components.model_component import build_model_candidates
from src.utils.paths import DEFAULT_DATASET

# Toggle pipeline execution. Set to True to regenerate preprocessing and reports.
RUN_PIPELINE = False
if RUN_PIPELINE:
    run_pipeline(skip_training=True)

# display the model candidates that will be evaluated
candidate_models = build_model_candidates()
print("Candidate classification models:")
for model_name in candidate_models:
    print(f"- {model_name}")
print()

# Train models using the shared project components and save the evaluation report.
report = train_models(DEFAULT_DATASET)

RESULTS_PATH = Path("../assets/reports/model_report_notebook.json")
RESULTS_PATH.parent.mkdir(parents=True, exist_ok=True)
RESULTS_PATH.write_text(json.dumps(report, indent=2), encoding="utf-8")

# Display the results sorted by F1 score for easier comparison.
results = pd.DataFrame([
    {
        "Model": name,
        "Accuracy": metrics.get("accuracy"),
        "Precision": metrics.get("precision"),
        "Recall": metrics.get("recall"),
        "F1": metrics.get("f1"),
    }
    for name, metrics in report["results"].items()
])
results = results.sort_values("F1", ascending=False).reset_index(drop=True)

print("Model training completed. Evaluation results sorted by F1 score:")
print(results.to_string(index=False))

best_model = results.iloc[0]
print(f"\nBest model: {best_model['Model']} with F1={best_model['F1']:.4f}")

In [ ]:
# Load an existing saved report and display the model performance table.
import json
import pandas as pd
from pathlib import Path

RESULTS_PATH = Path("../assets/reports/model_report_notebook.json")
report = json.loads(RESULTS_PATH.read_text(encoding="utf-8"))

results = pd.DataFrame([
    {
        "Model": name,
        "Accuracy": metrics.get("accuracy"),
        "Precision": metrics.get("precision"),
        "Recall": metrics.get("recall"),
        "F1": metrics.get("f1"),
    }
    for name, metrics in report["results"].items()
])
results = results.sort_values("F1", ascending=False).reset_index(drop=True)

print("Saved model report loaded. Performance metrics sorted by F1 score:")
print(results.to_string(index=False))